# AIE S4 — Forest Cover Type

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-ai-engineering/challenges/aie-s4-forest-cover.ipynb)

**Classification, 7 classes.** Predict which tree species dominates a
30 × 30 m patch of the Roosevelt National Forest, from 54
cartographic features.

Challenge: <https://ml-arena.com/viewchallenge/188>

---

**No code here, on purpose.** You have just worked through the
California-housing notebook, which built a PyTorch regressor from a
bare tensor up to a full training loop. This is the same machinery
pointed at a class label, and you write it.

Almost everything transfers unchanged: the tensors, the scaler, the
train/validation split, `DataLoader`, the five-line step, the two loss
curves, the best-checkpoint restore. **Three things change**, and this
notebook makes you find all three rather than telling you:

1. how wide the last layer is
2. which loss function
3. how you turn what the network outputs into an answer you can put
   in a CSV

Target: **beat the benchmark, accuracy 0.696.** A correctly built
54-128-64-7 MLP reaches 0.814.

---

## 0. Setup

Install the ML-Arena client. The distribution is **`mlarena-sdk`** and
it imports as `mlarena`; `pip install mlarena` is a different package.

`torch`, `pandas`, `scikit-learn` and `matplotlib` are already in
Colab.

---

## 1. Get the data

Connect with your `mlk_user_...` key and download the dataset for
challenge **188** into the working directory.

---

## 2. Read it, and answer three questions

Load the three files. Before you model anything, work out and write
down:

- What are **n** and **p**?
- **How many distinct classes are in `y_train`, and what are their
  values?** Print them. Do not assume.
- What accuracy would you get by always predicting the most common
  class? The classes here are balanced by construction, so you should
  be able to predict this number before computing it — then compute it
  and check.

The second question is not busy-work. The answer determines the shape
of your last layer, and it contains the trap in section 5.

---

## 3. A quick look

Three plots, no more — this is a PyTorch exercise, not an EDA one.

- The class balance, to confirm what section 2 told you.
- `elevation` split by cover type. Use a box plot or overlaid
  densities. This one column does a surprising amount of the work, and
  seeing that now will make a later number less mysterious.
- The **ranges** of the 54 columns — for instance
  `X_train.describe().loc[["min", "max"]].T`. Ten columns are real
  measurements and forty-four are 0/1 indicators. Note the widest and
  the narrowest, and predict what that will do to an unstandardised
  network before section 8 measures it.

---

## 4. From DataFrame to tensors

Same three steps as the regression notebook, same order, same reasons.

1. Split off a validation set.
2. `StandardScaler`, **fitted on the training part only**.
3. Convert to tensors.

Two differences from last time, and both matter:

**The target's dtype.** Regression targets were `float32` in a column
of shape `(n, 1)`. A classification target is a **class index**, and
`CrossEntropyLoss` requires `torch.long`, shape `(n,)` — a flat vector,
not a column. If you `.view(-1, 1)` this one you will get an error
about the target's dimensions.

**The off-by-one.** Your labels run 1–7. `CrossEntropyLoss` expects
class indices starting at **0**, so it wants 0–6. Subtract one on the
way in. Write it down now, because you will have to add it back in
section 9 and forgetting *that* is silent.

> Should you standardise the forty-four 0/1 indicator columns as well
> as the ten real ones? Try it both ways later and report which you
> chose. There is a defensible argument each way; the marks are for
> having an argument.

---

## 5. Build the model

Start from the California network:

```
Linear(8, 64) -> ReLU -> Linear(64, 64) -> ReLU -> Linear(64, 1)
```

and change what has to change. `Linear(54, 128) -> ReLU -> Linear(128,
64) -> ReLU -> Linear(64, ?)` is a good size for this data.

**How wide is the last layer, and why?** Answer that before you type
it.

**What does the network output now?** Not a probability. The last layer
is `nn.Linear`, so it emits an unbounded real number per class — these
are called **logits**. Larger means "more likely", and that is the only
guarantee.

> **Do not put a `nn.Softmax` at the end.** `nn.CrossEntropyLoss`
> applies log-softmax internally, so adding your own applies it twice.
> This does not raise. It trains — badly — and the usual symptom is a
> loss that falls for a while and then parks well above where it
> should. It is on Lab 4's automatic-deduction list for a reason.

Print your model and its parameter count, as the last notebook did.

---

## 6. The training loop

This is the part you should be able to write from memory now. It is
the California section 8 loop with the loss swapped:

- `TensorDataset` + `DataLoader`, `batch_size=128`, `shuffle=True`
- `nn.CrossEntropyLoss()`
- `torch.optim.Adam(model.parameters(), lr=1e-3)`
- 60 epochs of: `zero_grad` → forward → loss → `backward` → `step`
- `model.train()` and `model.eval()` in the right places
- a **validation loss every epoch**, under `torch.no_grad()`
- keep the best checkpoint and restore it at the end

Track **validation accuracy** per epoch as well as the loss. They are
not the same curve and the difference is instructive — the loss can
keep improving after accuracy has stopped, and vice versa.

Then plot the two losses on one pair of axes, exactly as before, and
read the plot with the same four rules.

> If your loss immediately becomes `nan`, or you get an `IndexError`
> mentioning a target out of bounds — that is the section 4
> off-by-one. Your labels still start at 1.

---

## 7. From logits to an answer

Your model outputs a `(n, 7)` tensor of logits. A submission needs one
integer per row.

- Which dimension do you reduce over, and what does `dim=1` mean here?
  Check the shape you get back.
- **Then add one.** `argmax` returns 0–6; cover types are 1–7. Forget
  this and every prediction is shifted by one class — accuracy near
  chance, and **no error anywhere**. The scorer rejects a `0` with a
  message about exactly this, which will catch you at upload time, but
  only if the shift pushed something to 0.

Report validation accuracy and **macro-F1** (`sklearn.metrics.f1_score`
with `average="macro"`) side by side. Also print a confusion matrix and
look at it: which two cover types does your model confuse, and does
that make ecological sense given the `elevation` plot from section 3?

---

## 8. Price the scaler

Repeat the California experiment on this data. Train the **identical**
network on **unstandardised** inputs — same architecture, same
optimizer, same 60 epochs, same best-checkpoint restore — and report
both validation accuracies.

The measured gap here is about eight points (0.814 against 0.737),
smaller than the
regression case. **Explain why it is smaller**, using what you found in
section 3. One sentence.

Then explain why it is not zero. `elevation` is in metres and `slope`
is in degrees; what does that do to the first layer's gradients?

---

## 9. Predict and submit

Transform `X_test` with the **same scaler** — the one fitted in section
4, not a new one. Predict, argmax, add one.

Before uploading, assert all four of these. Each corresponds to a real
way this goes wrong:

- one row per test id, and the ids are unique
- every value is in `{1, 2, 3, 4, 5, 6, 7}` — an integer, not a
  probability, not a numpy float that writes as `3.0`
- **more than one distinct class is predicted** (a collapsed head
  predicts two or three classes and nothing else)
- the column is named `prediction`, not `label`

That last one is worth a sentence. The MNIST warm-up (#182) uses
`id,label`; this challenge uses `id,prediction`. Upload validation
rejects the wrong header before anything runs.

Then submit, and compare against the benchmark on the challenge page
(accuracy 0.696).

---

## 10. Write down what you found

Five sentences, in the notebook:

- The three things you had to change from the regression notebook, and
  what each one would have done if you had got it wrong.
- Your validation accuracy and macro-F1, and your leaderboard accuracy.
  Which of the two do you believe, and why?
- What standardisation was worth here, and why less than on California
  housing.
- The two cover types your confusion matrix mixes up most, and whether
  that is a modelling failure or a property of the forest.
- One thing you tried that did not help.

That last one is not filler. A report with only successes describes a
process that did not happen.

---

**Check yourself before Lab 4.** You should now be able to write, from
an empty cell and without looking anything up: a `Dataset`/`DataLoader`,
an `nn.Sequential` MLP for either a number or a class, the five-line
training step, and an evaluation pass with `model.eval()` and
`torch.no_grad()`. Lab 4 assumes exactly that and adds packaging and
tests around it.